In [13]:
import csv
f = open("knapsack.csv")
csvfile = csv.DictReader(f, delimiter='\t')

headers = csvfile.fieldnames

table = []
for row in csvfile:
    table.append(row)
    
f.close()

In [14]:
print(headers)

['Project', 'Required Programmers', 'Required Capital', 'Estimated NPV']


In [15]:
table

[{'Project': '1',
  'Required Programmers': '7',
  'Required Capital': '250',
  'Estimated NPV': '650'},
 {'Project': '2',
  'Required Programmers': '6',
  'Required Capital': '175',
  'Estimated NPV': '550'},
 {'Project': '3',
  'Required Programmers': '9',
  'Required Capital': '300',
  'Estimated NPV': '600'},
 {'Project': '4',
  'Required Programmers': '5',
  'Required Capital': '150',
  'Estimated NPV': '450'},
 {'Project': '5',
  'Required Programmers': '6',
  'Required Capital': '145',
  'Estimated NPV': '375'},
 {'Project': '6',
  'Required Programmers': '4',
  'Required Capital': '160',
  'Estimated NPV': '525'},
 {'Project': '7',
  'Required Programmers': '8',
  'Required Capital': '325',
  'Estimated NPV': '750'},
 {'Project': 'Available',
  'Required Programmers': '20',
  'Required Capital': '950',
  'Estimated NPV': ''}]

In [12]:
for row in table:
    print(row)

{'Project': '1', 'Required Programmers': '7', 'Required Capital': '250', 'Estimated NPV': '650'}
{'Project': '2', 'Required Programmers': '6', 'Required Capital': '175', 'Estimated NPV': '550'}
{'Project': '3', 'Required Programmers': '9', 'Required Capital': '300', 'Estimated NPV': '600'}
{'Project': '4', 'Required Programmers': '5', 'Required Capital': '150', 'Estimated NPV': '450'}
{'Project': '5', 'Required Programmers': '6', 'Required Capital': '145', 'Estimated NPV': '375'}
{'Project': '6', 'Required Programmers': '4', 'Required Capital': '160', 'Estimated NPV': '525'}
{'Project': '7', 'Required Programmers': '8', 'Required Capital': '325', 'Estimated NPV': '750'}
{'Project': 'Available', 'Required Programmers': '20', 'Required Capital': '950', 'Estimated NPV': ''}


In [9]:
# read the parameters
ReqProgrammers = {}
ReqCapital = {}
EstNPV = {}
for row in table:
    if row['Project'] != 'Available':
        ReqProgrammers[row['Project']] = float(row['Required Programmers'])
        ReqCapital[row['Project']] = float(row['Required Capital'])
        EstNPV[row['Project']] = float(row['Estimated NPV'])
Budget = float(table[-1]['Required Capital'])
AvailableProgrammers = float(table[-1]['Required Programmers'])

In [10]:
ReqProgrammers

{'1': 7.0, '2': 6.0, '3': 9.0, '4': 5.0, '5': 6.0, '6': 4.0, '7': 8.0}

In [11]:
ReqCapital

{'1': 250.0,
 '2': 175.0,
 '3': 300.0,
 '4': 150.0,
 '5': 145.0,
 '6': 160.0,
 '7': 325.0}

In [12]:
EstNPV

{'1': 650.0,
 '2': 550.0,
 '3': 600.0,
 '4': 450.0,
 '5': 375.0,
 '6': 525.0,
 '7': 750.0}

In [13]:
Budget

950.0

In [14]:
AvailableProgrammers

20.0

In [26]:
from docplex.mp.model import Model
mdl = Model()

In [27]:
# variables
select = mdl.binary_var_dict(EstNPV, name='project')

In [28]:
# objective
mdl.maximize(mdl.sum(EstNPV[i]*select[i] for i in select))

In [29]:
# constraints
MaxProgrammers = mdl.add_constraint(mdl.sum(ReqProgrammers[i]*select[i] for i in select) <= AvailableProgrammers)
MaxBudget = mdl.add_constraint(mdl.sum(ReqCapital[i]*select[i] for i in select) <= Budget)

In [30]:
# solve
mdl.solve()
mdl.get_solve_details()

docplex.mp.SolveDetails(time=0.023051,status='integer optimal solution')

In [31]:
mdl.print_solution()

objective: 1925.000
status: OPTIMAL_SOLUTION(2)
  project_1=1
  project_6=1
  project_7=1


In [32]:
mdl.add_kpi(MaxProgrammers.lhs, 'Programmers used')
mdl.add_kpi(MaxBudget.lhs, 'Budget used')

DecisionKPI(name=Budget used,expr=250project_1+175project_2+300project_3+150project_4+145project_5..)

In [33]:
mdl.report()

* model docplex_model2 solved with objective = 1925.000
*  KPI: Programmers used = 19.000
*  KPI: Budget used      = 735.000
